# Giá lịch sử FireAnt

Nến NGÀY từ `restv2.fireant.vn/symbols/{ticker}/historical-quotes`, lưu vào `fscore.db`:

* `fireant_prices` — một dòng một phiên, khóa `(symbol, date)`
* `fireant_prices_meta` — mỗi mã một dòng: khoảng ngày, số phiên, `fetched_at`

Khác Simplize (lịch sử dài chỉ có nến tháng, giá đã điều chỉnh sẵn), FireAnt trả nến ngày cho
toàn bộ lịch sử và giữ **giá thô** (`price_close`) kèm **hệ số điều chỉnh** (`adj_ratio`),
nên tự tính lại được giá điều chỉnh:

* giá VND = `price * unit` (`unit` = 1000 với cổ phiếu, 1 với chỉ số)
* giá điều chỉnh = `price * unit / adj_ratio`
* khối lượng điều chỉnh = `volume * adj_ratio`

`adj_ratio` là hệ số **luỹ kế tới ngày fetch** (bằng 1.0 ở phiên gần nhất), nên một đợt chia tách
mới làm lệch toàn bộ adj_ratio cũ trong DB → xem `fetched_at` trong bảng meta, muốn đồng bộ lại
thì crawl với `mode='full'`.

In [ ]:
from fireant_price_connector import FireantPrices
import pandas as pd

tickers = pd.read_csv("./data/fireant_grid_sorted.csv")
tickers = tickers[tickers["coverage_score"] >= 6]
tickers

In [ ]:
# chỉ số crawl trước để có benchmark ngay cả khi đợt crawl bị đứt giữa chừng
INDICES = ['VNINDEX', 'VN30', 'HNXINDEX', 'HNX30', 'UPINDEX']
symbols = INDICES + tickers["Unnamed: 0"].tolist()
len(symbols)

In [ ]:
fa = FireantPrices(request_sleep=0.2)
# mode='skip' bỏ qua mã đã có trong DB -> chạy lại được sau khi đứt giữa chừng
log = fa.crawl_to_db(symbols, start_date='2009-01-01', db_path='fscore.db',
                     sleep=0.3, mode='skip', log_path='logs/fireant_prices.csv')
log

In [ ]:
# mã lỗi / mã không có dữ liệu
log[~log['IsSuccess'] | ((log['Bars'] == 0) & log['ErrorMessage'].isna())]

## Kiểm tra dữ liệu đã lưu

In [ ]:
meta = FireantPrices.load_meta('fscore.db')
print(len(meta), 'mã |', meta['bars'].sum(), 'phiên')
meta

In [ ]:
# adjusted=True thêm các cột *_adj (giá VND đã điều chỉnh, khối lượng đã điều chỉnh)
hpg = FireantPrices.load_prices('fscore.db', symbol='HPG', adjusted=True)
hpg[['date', 'price_close', 'unit', 'adj_ratio', 'price_close_adj',
     'total_volume', 'total_volume_adj']]

In [ ]:
# các ngày adj_ratio đổi = ngày GDKHQ (chia tách / cổ tức)
hpg.loc[hpg['adj_ratio'].diff().fillna(0) != 0,
        ['date', 'price_basic', 'price_close', 'adj_ratio']]

## Cập nhật thêm phiên mới

`mode='update'` chỉ lấy từ phiên cuối đã có trong DB (adj_ratio của phiên cũ giữ nguyên).
Nếu trong kỳ có mã chia tách thì dùng `mode='full'` cho mã đó để lấy lại adj_ratio đồng bộ.

In [ ]:
# log_update = fa.crawl_to_db(symbols, start_date='2009-01-01', db_path='fscore.db',
#                             sleep=0.3, mode='update',
#                             log_path='logs/fireant_prices_update.csv')
# log_update